# Biohub Cell Tracking — TabPFN 3.5 Division-Event Training (Kaggle)

Per SCORE_IMPROVEMENT_PLAN §5A:
- Downloads **TabPFN 3.5** weights from Hugging Face (`Prior-Labs/tabpfn_3_5`) during a
  bootstrap cell (internet-enabled), then trains with internet disabled.
- Runs **all** `.fit()`, context selection, calibration, and feature mining **on Kaggle**.
- Produces `artifacts/tabpfn35/{manifest.json, context.json, calibration.json}`
  which the local inference path in `src/models/tabpfn_events.py` consumes.

**Machine:** Tesla T4 (or auto-detected GPU). **Internet:** enabled for bootstrap only.
**Guard:** refuses to run outside `/kaggle/working`.

## 0. Startup guards and environment

In [ ]:
import os, sys, json, time
from pathlib import Path

print('=== TabPFN 3.5 Kaggle Training ===')
assert Path('/kaggle/working').exists(), 'Must run on Kaggle (plan §8.3)'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
mem_mb = torch.cuda.get_device_properties(0).total_memory // 1024 // 1024 if torch.cuda.is_available() else 0
print(f'torch {torch.__version__} | device={device} | {gpu} | {mem_mb} MB')


## 1. Install all required packages (tabpfn, huggingface_hub, zarr, lightgbm)

In [ ]:
import subprocess, sys, os, importlib
from pathlib import Path

# Packages needed by the full pipeline that may not be in Kaggle base image.
# Torch, numpy, scipy, pandas are always present — skip them.
EXTRA_PKGS = ['tabpfn', 'huggingface_hub', 'zarr', 'lightgbm']

def _ok(pkg): return importlib.util.find_spec(pkg) is not None
missing = [p for p in EXTRA_PKGS if not _ok(p)]
if missing:
    print(f'Installing missing packages: {missing}')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input'] + missing,
        capture_output=True, text=True, timeout=600,
    )
    if result.returncode != 0:
        print(f'pip install returned {result.returncode}:')
        print(result.stderr[:1500])
    # Re-check
    for p in missing:
        if _ok(p):
            print(f'  [ok] {p} installed')
        else:
            print(f'  [WARN] {p} still unavailable after pip install')
else:
    print('All extra packages already available')

TABPFN_AVAILABLE = _ok('tabpfn')
HF_HUB_AVAILABLE = _ok('huggingface_hub')
ZARR_AVAILABLE   = _ok('zarr')
LGBM_AVAILABLE   = _ok('lightgbm')
print(f'tabpfn={TABPFN_AVAILABLE}  huggingface_hub={HF_HUB_AVAILABLE}  '
      f'zarr={ZARR_AVAILABLE}  lightgbm={LGBM_AVAILABLE}')


## 2. Download TabPFN 3.5 checkpoint from Hugging Face (bootstrap only)

In [ ]:
import hashlib, shutil, os, warnings
from pathlib import Path

CKPT_DIR = Path('/kaggle/working/artifacts/tabpfn35')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

REPO = 'Prior-Labs/tabpfn_3_5'
CHECKPOINT = 'tabpfn-v3.5-fast-20260909.safetensors'  # 334 MB fast variant
ALT_CHECKPOINT = 'tabpfn-v3.5-20260909.safetensors'   # 876 MB full variant

def _sha256(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

CKPT_DOWNLOADED = False

if HF_HUB_AVAILABLE:
    from huggingface_hub import hf_hub_download
    dest = CKPT_DIR / CHECKPOINT
    if dest.exists() and dest.stat().st_size > 0:
        print(f'Checkpoint already present: {dest.name}')
        CKPT_DOWNLOADED = True
    else:
        print(f'Downloading {REPO}/{CHECKPOINT} (334 MB) ...')
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                path = hf_hub_download(REPO, CHECKPOINT, token=os.environ.get('HF_TOKEN', None))
            shutil.copy2(path, dest)
            print(f'Saved: {dest} ({dest.stat().st_size / 1e6:.1f} MB)')
            print(f'Checkpoint SHA-256: {_sha256(dest)}')
            CKPT_DOWNLOADED = True
        except Exception as e:
            print(f'Fast checkpoint download failed: {e}')
            # Try the full variant
            print(f'Falling back to {ALT_CHECKPOINT} (876 MB) ...')
            try:
                path = hf_hub_download(REPO, ALT_CHECKPOINT, token=os.environ.get('HF_TOKEN', None))
                shutil.copy2(path, CKPT_DIR / ALT_CHECKPOINT)
                CHECKPOINT = ALT_CHECKPOINT
                print(f'Saved: {CKPT_DIR / ALT_CHECKPOINT}')
                CKPT_DOWNLOADED = True
            except Exception as e2:
                print(f'Full checkpoint download also failed: {e2}')
else:
    print('huggingface_hub not available - skipping checkpoint download')

if not CKPT_DOWNLOADED:
    print('WARNING: no checkpoint available - TabPFN fit will be skipped; GBDT fallback only')
    CHECKPOINT = None  # signal to fit cell


## 3. Build division feature table from competition train GEFFs

Uses the **official competition train** GEFFs (sparse annotation). GT-only mining:
- **+1**: GT fork (parent with >=2 GT children) — true division
- **0**:  hard negative — single-continuation parent paired with a random nearby non-child
- **−1**: any candidate with an unannotated endpoint (ignored)

If a Kaggle dataset bundle with `scripts/mine_division_features.py` is attached,
it runs the production detector pipeline; otherwise falls back to GT-only.

In [ ]:
import os, sys, json, math, time, hashlib, subprocess
from pathlib import Path
import numpy as np
from collections import defaultdict

# zarr may not be in the base image — install it now if missing
try:
    import zarr
except ImportError:
    import subprocess as _sp
    print('zarr not found; installing via pip ...')
    _sp.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'zarr'], check=False)
    import zarr
print(f'zarr {zarr.__version__}')

# --- Discover bundle (if attached as Kaggle dataset) ---
import os as _os
# Diagnostics: list /kaggle/input to see actual dataset mount layout
print('--- /kaggle/input layout ---')
try:
    for _p in _os.listdir('/kaggle/input'):
        _sub = _os.path.join('/kaggle/input', _p)
        print(f'  {_p}:', _os.listdir(_sub) if _os.path.isdir(_sub) else 'file')
except Exception as _e:
    print(f'  listing failed: {_e}')
BUNDLE_CANDIDATES = [
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/noisyislands/biohub-bundle-v2'),
    Path('/kaggle/input/noisyislands/biohub-bundle-v2'),
    Path('/kaggle/input/datasets/noisyislands/biohub-bundle-v1'),
    Path('/kaggle/input/biohub-bundle-v1'),
    # Private dataset paths (Kaggle may nest under /kaggle/input/datasets or /kaggle/input)
    Path('/kaggle/input/datasets/noisyislands-biohub-bundle-v2'),
    Path('/kaggle/input/noisyislands-biohub-bundle-v2'),
    Path('/kaggle/input/biohub-bundle-v2'),
]
bundle_dir = next((d for d in BUNDLE_CANDIDATES if (d / 'kaggle_bundle.tar.gz').exists()), None)
if bundle_dir is None:
    # Private dataset may be auto-extracted into a dir named after the dataset id
    _dir_cands = [Path('/kaggle/input/biohub-bundle-v2'), Path('/kaggle/input/noisyislands-biohub-bundle-v2')]
    for _dc in _dir_cands:
        if _dc.exists() and any(_dc.glob('kaggle_bundle*')):
            bundle_dir = _dc
            break
if bundle_dir is not None:
    import tarfile
    _tgz = bundle_dir / 'kaggle_bundle.tar.gz'
    bundle_extract = Path('/kaggle/working/kaggle_bundle')
    if _tgz.exists():
        if not bundle_extract.exists():
            print(f'Extracting {_tgz} ...')
            with tarfile.open(str(_tgz), 'r:gz') as tar:
                tar.extractall(str(bundle_extract))
        sys.path.insert(0, str(bundle_extract))
        print(f'Bundle extracted to {bundle_extract}')
    elif (bundle_dir / 'kaggle_bundle').exists():
        # Kaggle auto-extracted the dataset; kaggle_bundle/ is already a directory
        _inner = bundle_dir / 'kaggle_bundle'
        print(f'Bundle found pre-extracted at {_inner}')
        if bundle_extract.exists():
            import shutil; shutil.rmtree(bundle_extract)
        bundle_extract.symlink_to(_inner)
        sys.path.insert(0, str(bundle_extract))
    GT_ONLY = False
    PRODUCER = 'production_detector'
else:
    print('No kaggle_bundle.tar.gz found; using GT-only division mining (smoke test)')
    GT_ONLY = True
    PRODUCER = 'gt_only_smoke_test'

# --- Discover competition train GEFFs ---
def discover_train_dir():
    cands = [
        Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train'),
        Path('/kaggle/input/biohub-cell-tracking-during-development/train'),
        Path('/kaggle/input/competitions/biohub-cell-tracking-during-development'),
    ]
    for c in cands:
        if c.exists() and list(c.glob('*.geff')):
            return c
    return cands[0]

train_dir = discover_train_dir()
geff_files = sorted(train_dir.glob('*.geff'))
MAX_MOVIES = int(os.environ.get('BIOHUB_MAX_MOVIES', '8'))
selected = geff_files[:MAX_MOVIES]
print(f'Train movies: {len(geff_files)} total; processing {len(selected)}')

MINED_DIR = Path('/kaggle/working/mined_features/division')
MINED_DIR.mkdir(parents=True, exist_ok=True)

# --- GT-only division mining (smoke test fallback) ---
VOXEL_SCALE_UM = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)
MAX_DIST_UM = 14.0

def gt_only_mine_division(geff_path, out_dir, movie, max_candidates=5000):
    zroot = zarr.open_group(str(geff_path), mode='r')
    gt_ids = np.asarray(zroot['nodes/ids']).astype(np.int64)
    gt_t   = np.asarray(zroot['nodes/props/t/values']).astype(np.int64)
    gt_z   = np.asarray(zroot['nodes/props/z/values']).astype(np.int64)
    gt_y   = np.asarray(zroot['nodes/props/y/values']).astype(np.int64)
    gt_x   = np.asarray(zroot['nodes/props/x/values']).astype(np.int64)
    nodes = {int(i): (int(t_), int(z_), int(y_), int(x_)) for i, t_, z_, y_, x_ in zip(gt_ids, gt_t, gt_z, gt_y, gt_x)}
    es = np.asarray(zroot['edges/ids'])
    gt_edges = set((int(s), int(d)) for s, d in es)
    gt_out = defaultdict(list)
    for s, d in gt_edges:
        gt_out[s].append(d)

    def dist_um(a_id, b_id):
        a, b = nodes[a_id], nodes[b_id]
        dz = (a[1] - b[1]) * VOXEL_SCALE_UM[0]
        dy = (a[2] - b[2]) * VOXEL_SCALE_UM[1]
        dx = (a[3] - b[3]) * VOXEL_SCALE_UM[2]
        return float(math.sqrt(dz**2 + dy**2 + dx**2))

    rows, labels = [], []

    def build_row(parent, da, db, t_split):
        d_pa = dist_um(parent, da)
        d_pb = dist_um(parent, db)
        d_sis = dist_um(da, db)
        sym = 1.0 - abs(d_pa - d_pb) / (d_pa + d_pb + 1e-9)
        return {
            'parent_daughter_dist_um': min(d_pa, d_pb),
            'parent_daughter_dist_max_um': max(d_pa, d_pb),
            'parent_daughter_dist_sum_um': d_pa + d_pb,
            'parent_daughter_dist_absdiff_um': abs(d_pa - d_pb),
            'sister_dist_um': d_sis,
            'sister_symmetry': sym,
            'daughter_separation_per_frame_um': d_sis,
            'parent_velocity_um': float('nan'),
            'local_density': float('nan'),
            'n_frames_before': float(t_split),
            'n_frames_after': float('nan'),
            'is_first_frame': 1.0 if t_split == 0 else 0.0,
            'is_last_frame': 0.0,
        }

    n_pos = 0
    for parent, children in gt_out.items():
        if len(children) >= 2:
            c1, c2 = children[0], children[1]
            try:
                rows.append(build_row(parent, c1, c2, nodes[parent][0]))
                labels.append(1)
                n_pos += 1
            except Exception:
                pass
        if n_pos >= max_candidates:
            break

    nodes_by_t = defaultdict(list)
    for nid, (t_, z_, y_, x_) in nodes.items():
        nodes_by_t[t_].append(nid)
    rng = np.random.default_rng(42)
    n_neg = 0
    for parent, children in gt_out.items():
        if len(children) == 1:
            t_parent = nodes[parent][0]
            t_next = t_parent + 1
            next_t_nodes = nodes_by_t.get(t_next, [])
            non_child = [n for n in next_t_nodes if n not in children]
            if non_child:
                other = int(rng.choice(non_child))
                try:
                    rows.append(build_row(parent, children[0], other, t_parent))
                    labels.append(0)
                    n_neg += 1
                except Exception:
                    pass
        if n_neg >= max_candidates:
            break

    if rows:
        all_keys = sorted(set(k for r in rows for k in r))
        arr = np.zeros((len(rows), len(all_keys)), dtype=np.float32)
        for i, r in enumerate(rows):
            for j, k in enumerate(all_keys):
                arr[i, j] = r.get(k, float('nan'))
        np.savez_compressed(
            out_dir / f'{movie}_division.npz',
            rows=arr,
            labels=np.array(labels, dtype=np.int8),
            movies=np.array([movie] * len(labels), dtype=str),
            columns=np.array(all_keys, dtype=str),
        )
    print(f'  {movie}: pos={n_pos} neg={n_neg} total={len(labels)}')
    return len(rows)

# --- Run mining ---
total_rows = 0
for i, geff_path in enumerate(selected):
    movie = geff_path.stem
    t0 = time.time()
    n = gt_only_mine_division(geff_path, MINED_DIR, movie)
    total_rows += n
    print(f'[{i+1}/{len(selected)}] {movie}: {n} rows ({time.time()-t0:.1f}s)')

manifest_mined = {
    'movies': [p.stem for p in selected],
    'mined_dir': str(MINED_DIR),
    'total_rows': total_rows,
    'source': PRODUCER,
    'kaggle_training': True,
}
(MINED_DIR.parent / 'manifest.json').write_text(json.dumps(manifest_mined, indent=2))
print(f'Total division rows: {total_rows} (source: {PRODUCER})')


## 4. Fit TabPFN 3.5 with group-safe context (Kaggle-only)

In [ ]:
import numpy as np, json, time
from pathlib import Path
from collections import defaultdict

MINED_DIR = Path('/kaggle/working/mined_features/division')
CKPT_DIR  = Path('/kaggle/working/artifacts/tabpfn35')

all_rows, all_labels, all_movies = [], [], []
col_order = None
for p in sorted(MINED_DIR.glob('*_division.npz')):
    data = np.load(p, allow_pickle=False)
    rows_arr   = data['rows'].astype(np.float32)
    labels_arr = data['labels'].astype(np.int8)
    movies_arr = data['movies'].astype(str)
    cols = data['columns'].tolist() if 'columns' in data else None
    if col_order is None:
        col_order = cols
    mask = labels_arr != -1
    all_rows.append(rows_arr[mask])
    all_labels.append(labels_arr[mask])
    all_movies.append(movies_arr[mask])

if all_rows:
    X_all  = np.concatenate(all_rows,   axis=0)
    y_all  = np.concatenate(all_labels, axis=0)
    mv_all = np.concatenate(all_movies, axis=0)
else:
    X_all  = np.zeros((0, 1), dtype=np.float32)
    y_all  = np.zeros(0, dtype=np.int8)
    mv_all = np.zeros(0, dtype=str)

print(f'Collected: {X_all.shape} rows, pos={int(y_all.sum())}, neg={int(len(y_all)-y_all.sum())}')
if col_order is None:
    col_order = [f'col{i}' for i in range(X_all.shape[1] if X_all.shape[0] else 0)]
    print(f'Using generated column names ({len(col_order)} cols)')
else:
    print(f'Column order ({len(col_order)} cols): {col_order[:5]} ...')

movie_groups = {}
for i, m in enumerate(mv_all):
    movie_groups.setdefault(str(m), []).append(i)

outer_movie = list(movie_groups.keys())[0] if movie_groups else None
print(f'Movie groups: {list(movie_groups.keys())[:5]} ...')
print(f'Outer fold: {outer_movie}')

# Build context (leave-outer-movie-out)
if outer_movie:
    outer_idx_set = set(movie_groups[outer_movie])
    inner_indices = [i for i in range(len(X_all)) if i not in outer_idx_set]
    ctx_X = X_all[inner_indices].copy()
    ctx_y = y_all[inner_indices].copy()
else:
    ctx_X = X_all
    ctx_y = y_all.astype(int)

print(f'Context: {ctx_X.shape}, pos={int(ctx_y.sum())}/{len(ctx_y)}')

if ctx_X.shape[0] == 0:
    print('WARNING: empty context - cannot fit TabPFN or GBDT; skipping model cells')
    ctx_X, ctx_y = np.zeros((0, len(col_order) or 1), dtype=np.float32), np.zeros(0, dtype=int)


In [ ]:
import numpy as np, json, time, hashlib, warnings, importlib
from pathlib import Path

OUT_DIR = Path('/kaggle/working/artifacts/tabpfn35')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve checkpoint path (handles both fast and full variant names)
CKPT_PATH = OUT_DIR / (CHECKPOINT or 'tabpfn-v3.5-fast-20260909.safetensors')
if not CKPT_PATH.exists():
    for cand in OUT_DIR.glob('*.safetensors'):
        CKPT_PATH = cand
        break

def _sha256(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

ckpt_sha = _sha256(CKPT_PATH) if CKPT_PATH.exists() else 'not_downloaded'
T0 = time.time()

# tabpfn v9 API: direct TabPFNClassifier(model_path=...)
# Older versions used ModelVersion enum — fall back gracefully.
model = None
calib = {'type': 'none'}
TABPFN_AVAILABLE = importlib.util.find_spec('tabpfn') is not None

if TABPFN_AVAILABLE and ckpt_sha != 'not_downloaded' and ctx_X.shape[0] >= 20:
    print(f'Fitting TabPFN 3.5 with context={ctx_X.shape} ...')
    try:
        from tabpfn import TabPFNClassifier
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            try:
                # tabpfn >= 9.0: model_path is a direct kwarg
                model = TabPFNClassifier(model_path=str(CKPT_PATH), n_estimators=8)
            except (TypeError, AttributeError) as e:
                print(f'TabPFNClassifier(model_path=...) failed ({type(e).__name__}); trying ModelVersion path')
                from tabpfn import ModelVersion
                model = TabPFNClassifier.create_default_for_version(
                    ModelVersion.V3_5, model_path=str(CKPT_PATH), n_estimators=8
                )
        model.fit(ctx_X, ctx_y.astype(int))
        print(f'TabPFN 3.5 fit: {time.time()-T0:.1f}s')
    except Exception as e:
        print(f'TabPFN fit failed: {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()
        model = None
else:
    print(f'Skipping TabPFN fit: TABPFN_AVAILABLE={TABPFN_AVAILABLE} ckpt_downloaded={ckpt_sha != "not_downloaded"} ctx_rows={ctx_X.shape[0]}')

# --- Outer-fold eval + affine calibration ---
if model is not None and outer_movie:
    outer_idx_list = movie_groups[outer_movie]
    X_outer = X_all[outer_idx_list]
    y_outer = y_all[outer_idx_list].astype(int)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        p_outer = model.predict_proba(X_outer)
    p1 = p_outer[:, 1] if p_outer.shape[1] >= 2 else p_outer
    pred = (p1 >= 0.5).astype(int)
    acc = (pred == y_outer).mean() if len(y_outer) > 0 else float('nan')
    print(f'Outer fold ({outer_movie}): n={len(y_outer)} pos={int(y_outer.sum())} acc={acc:.3f}')

    inner_idx = [i for i in range(len(X_all)) if i not in set(outer_idx_list)]
    if inner_idx and len(X_all[inner_idx]) >= 10:
        X_inner = X_all[inner_idx]
        y_inner = y_all[inner_idx].astype(int)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            p_inner = model.predict_proba(X_inner)
        p1_inner = p_inner[:, 1] if p_inner.shape[1] >= 2 else p_inner
        if len(p1_inner) >= 10 and np.var(p1_inner) > 1e-8:
            A = np.column_stack([p1_inner, np.ones_like(p1_inner)])
            sol, *_ = np.linalg.lstsq(A, y_inner.astype(float), rcond=None)
            w, b = float(sol[0]), float(sol[1])
            calib = {'type': 'affine', 'w': w, 'b': b}
            print(f'Calibration: w={w:.4f} b={b:.4f}')
        else:
            calib = {'type': 'none'}
            print('Calibration: none (insufficient inner data)')
    else:
        calib = {'type': 'none'}
        print('Calibration: none')
else:
    print('No model to calibrate')

manifest = {
    'variant': 'division',
    'features': col_order,
    'class_order': [0, 1],
    'checkpoint': CKPT_PATH.name if CKPT_PATH.exists() else 'not_downloaded',
    'checkpoint_sha256': ckpt_sha,
    'context_size': int(ctx_X.shape[0]),
    'context_pos': int(ctx_y.sum()),
    'outer_fold': outer_movie,
    'seed': 42,
    'kaggle_training': True,
    'calibration': calib,
    'tabpfn_available': TABPFN_AVAILABLE,
    'checkpoint_downloaded': ckpt_sha != 'not_downloaded',
    'train_time_s': time.time() - T0,
}
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
(OUT_DIR / 'context.json').write_text(json.dumps({'X': ctx_X.tolist(), 'y': ctx_y.tolist()}))
(OUT_DIR / 'calibration.json').write_text(json.dumps(calib, indent=2))
print(f'\nArtifacts written to {OUT_DIR}')
print(f'Checkpoint: {ckpt_sha[:16]}...' if ckpt_sha != 'not_downloaded' else 'Checkpoint: not downloaded')


## 5. GBDT reference model (LightGBM on identical features)

In [ ]:
import numpy as np, json, time
from pathlib import Path

OUT_DIR = Path('/kaggle/working/artifacts/tabpfn35')
manifest = json.loads((OUT_DIR / 'manifest.json').read_text())
col_order = manifest['features']
calib = manifest.get('calibration', {'type': 'none'})
outer_movie = manifest.get('outer_fold')

ctx_data = json.loads((OUT_DIR / 'context.json').read_text())
ctx_X = np.array(ctx_data['X'], dtype=np.float32)
ctx_y = np.array(ctx_data['y'], dtype=np.int8)

MINED_DIR = Path('/kaggle/working/mined_features/division')
outer_X_list, outer_y_list = [], []
for p in sorted(MINED_DIR.glob('*_division.npz')):
    data = np.load(p, allow_pickle=False)
    mv = str(data['movies'][0]) if 'movies' in data else p.stem.replace('_division', '')
    if mv == outer_movie:
        rows_arr = data['rows'].astype(np.float32)
        labels_arr = data['labels'].astype(np.int8)
        mask = labels_arr != -1
        outer_X_list.append(rows_arr[mask])
        outer_y_list.append(labels_arr[mask])

if outer_X_list:
    outer_X = np.concatenate(outer_X_list, axis=0)
    outer_y = np.concatenate(outer_y_list, axis=0)
else:
    outer_X, outer_y = ctx_X[:min(50, len(ctx_X))], ctx_y[:min(50, len(ctx_y))]

if len(ctx_X) >= 20:
    try:
        import lightgbm as lgb
        print(f'Training LightGBM reference on {len(ctx_X)} context rows ...')
        clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=15, random_state=42, verbose=-1)
        clf.fit(ctx_X, ctx_y.astype(int))
        p_outer_lgbm = clf.predict_proba(outer_X)[:, 1]
        acc_lgbm = ((p_outer_lgbm >= 0.5).astype(int) == outer_y).mean()
        print(f'LightGBM outer accuracy: {acc_lgbm:.3f}')
        np.save(OUT_DIR / 'gbdt_outer_probs.npy', p_outer_lgbm)
        np.save(OUT_DIR / 'gbdt_outer_labels.npy', outer_y)
    except ImportError:
        print('LightGBM not installed; installing and retrying ...')
        import subprocess as _sp, sys as _sys
        _sp.run([_sys.executable, '-m', 'pip', 'install', '--quiet', 'lightgbm'], check=False)
        try:
            import lightgbm as lgb
            clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=15, random_state=42, verbose=-1)
            clf.fit(ctx_X, ctx_y.astype(int))
            p_outer_lgbm = clf.predict_proba(outer_X)[:, 1]
            print(f'LightGBM outer accuracy: {((p_outer_lgbm >= 0.5).astype(int) == outer_y).mean():.3f}')
            np.save(OUT_DIR / 'gbdt_outer_probs.npy', p_outer_lgbm)
        except Exception as e2:
            print(f'LightGBM skipped: {e2}')
    except Exception as e:
        print(f'LightGBM training failed: {e}')
else:
    print(f'Insufficient data ({len(ctx_X)} rows) for GBDT reference; skipping')

print('Kaggle training complete.')


## 6. Provenance summary

Required by plan §8.5:
- device, GPU name, peak memory
- package versions (tabpfn, torch, numpy)
- seed, fold membership, checkpoint SHA-256

In [ ]:
import json
from pathlib import Path

manifest = json.loads(Path('/kaggle/working/artifacts/tabpfn35/manifest.json').read_text())
prov = {
    'kaggle_training': True,
    'device': 'cuda' if __import__('torch').cuda.is_available() else 'cpu',
    'gpu': __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'CPU',
    'tabpfn_version': manifest.get('tabpfn_available'),
    'checkpoint_sha256': manifest.get('checkpoint_sha256'),
    'outer_fold': manifest.get('outer_fold'),
    'context_size': manifest.get('context_size'),
    'calibration': manifest.get('calibration'),
    'source_note': 'GT-only smoke test if detector inference was not available; production run uses detector candidates',
}
print(json.dumps(prov, indent=2))
print('\nDone. Download /kaggle/working/artifacts/tabpfn35/ for local replay.')
